# 01. Ingeniería de datos — LaLiga

Aquí construyo `df_final_clean.csv` partiendo de los CSV brutos de football-data.co.uk. Le añado ratings FIFA y valores de mercado de Transfermarkt, y calculo unas cuantas variables propias: Elo, rachas, H2H y diferencias de plantilla.

%pip install pandas numpy


In [1]:
%pip install pandas numpy

import pandas as pd
import numpy as np
import json, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
OUT_CSV  = '../data/df_final_clean.csv'

# Carga y concatenacion de todas las temporadas SP1
import glob
files = sorted(glob.glob(os.path.join(DATA_DIR, 'SP1_*.csv')))
dfs = []
skipped = []
for f in files:
    season_tag = os.path.basename(f).replace('SP1_', '').replace('.csv', '')
    try:
        df_s = pd.read_csv(f, encoding='latin1')
        df_s['source_file'] = season_tag
        dfs.append(df_s)
    except Exception as e:
        skipped.append(os.path.basename(f))
        print(f'  ⚠️  AVISO — archivo ignorado: {os.path.basename(f)} → {e}')

if skipped:
    print(f'\n⚠️  {len(skipped)} archivo(s) no cargado(s): {skipped}')
    print('   Los partidos de esas temporadas NO están en el dataset.\n')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'Partidos totales: {len(df_raw)}')
print(f'Columnas: {list(df_raw.columns[:15])}')

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\emili\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


  ⚠️  AVISO — archivo ignorado: SP1_0405.csv → Error tokenizing data. C error: Expected 49 fields in line 97, saw 50


⚠️  1 archivo(s) no cargado(s): ['SP1_0405.csv']
   Los partidos de esas temporadas NO están en el dataset.

Partidos totales: 9410
Columnas: ['Div', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'GBH', 'GBD', 'GBA', 'IWH', 'IWD']


## 1. Limpieza básica

In [2]:
# Columnas minimas necesarias
REQUIRED = ['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR']
df = df_raw.dropna(subset=REQUIRED).copy()

# Fechas
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Temporada: agosto-mayo
def get_season(d):
    return d.year if d.month >= 8 else d.year - 1

df['Season'] = df['Date'].apply(get_season)

# Solo temporadas 2010-2024 (datos de cuotas completos)
df = df[(df['Season'] >= 2010) & (df['Season'] <= 2024)]
df = df[df['FTR'].isin(['H','D','A'])]

# Target encoding: H=2, D=1, A=0
df['Target'] = df['FTR'].map({'H': 2, 'D': 1, 'A': 0})

print(f'Partidos tras limpieza: {len(df)}')
print(f'Temporadas: {sorted(df["Season"].unique())}')
print(df.groupby('Season').size().rename('partidos'))


Partidos tras limpieza: 5700
Temporadas: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Season
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
Name: partidos, dtype: int64


## 2. Elo calculado desde cero

In [3]:
# Elo con ventaja de campo (HA=100) y K=30
ELO_K, ELO_HA, ELO_START = 30, 100, 1500

ratings = {}
h_elos, a_elos = [], []

for _, row in df.iterrows():
    h, a, ftr = row['HomeTeam'], row['AwayTeam'], row['FTR']
    rh = ratings.get(h, ELO_START)
    ra = ratings.get(a, ELO_START)
    h_elos.append(rh)
    a_elos.append(ra)
    e_h = 1.0 / (1.0 + 10 ** ((ra - (rh + ELO_HA)) / 400.0))
    e_a = 1.0 - e_h
    s_h, s_a = (1, 0) if ftr == 'H' else ((0.5, 0.5) if ftr == 'D' else (0, 1))
    ratings[h] = rh + ELO_K * (s_h - e_h)
    ratings[a] = ra + ELO_K * (s_a - e_a)

df['Home_Elo_Calc'] = h_elos
df['Away_Elo_Calc'] = a_elos
df['Elo_Diff']      = df['Home_Elo_Calc'] - df['Away_Elo_Calc']

print(f'Elo std local: {df["Home_Elo_Calc"].std():.1f} '
      f'(si ~0 indica error; deberia ser >80)')


Elo std local: 113.3 (si ~0 indica error; deberia ser >80)


## 3. Variables de forma (rachas y H2H)

In [4]:
# Racha de victorias/derrotas de los ultimos 5 partidos
def compute_streaks(df):
    team_results = {}
    h_streak, a_streak = [], []
    for _, row in df.iterrows():
        h, a, ftr = row['HomeTeam'], row['AwayTeam'], row['FTR']
        # Racha acumulada: +1 por victoria, -1 por derrota, 0 por empate
        hr = team_results.get(h, [])
        ar = team_results.get(a, [])
        h_streak.append(sum(hr[-5:]))
        a_streak.append(sum(ar[-5:]))
        # Actualizar resultados
        if ftr == 'H':
            team_results[h] = hr + [1]
            team_results[a] = ar + [-1]
        elif ftr == 'A':
            team_results[h] = hr + [-1]
            team_results[a] = ar + [1]
        else:
            team_results[h] = hr + [0]
            team_results[a] = ar + [0]
    return h_streak, a_streak

df['Home_Streak_L5'], df['Away_Streak_L5'] = compute_streaks(df)

# H2H ultimos 3 enfrentamientos entre esos dos equipos
def compute_h2h(df):
    h2h_hist = {}
    h_h2h, a_h2h = [], []
    for _, row in df.iterrows():
        h, a, ftr = row['HomeTeam'], row['AwayTeam'], row['FTR']
        key = tuple(sorted([h, a]))
        hist = h2h_hist.get(key, [])
        # Desde perspectiva del local
        h_score = sum(1 if r == h else (-1 if r == a else 0) for r in hist[-3:])
        a_score = -h_score
        h_h2h.append(h_score)
        a_h2h.append(a_score)
        winner = h if ftr == 'H' else (a if ftr == 'A' else 'D')
        h2h_hist[key] = hist + [winner]
    return h_h2h, a_h2h

df['Home_H2H_L3'], df['Away_H2H_L3'] = compute_h2h(df)
print('Rachas y H2H calculados.')
print(df[['Home_Streak_L5','Away_Streak_L5','Home_H2H_L3','Away_H2H_L3']].describe().round(2))


Rachas y H2H calculados.
       Home_Streak_L5  Away_Streak_L5  Home_H2H_L3  Away_H2H_L3
count         5700.00         5700.00      5700.00      5700.00
mean            -0.09            0.11        -0.07         0.07
std              2.19            2.19         1.54         1.54
min             -5.00           -5.00        -3.00        -3.00
25%             -2.00           -1.00        -1.00        -1.00
50%              0.00            0.00         0.00         0.00
75%              1.00            2.00         1.00         1.00
max              5.00            5.00         3.00         3.00


## 4. Variables externas: FIFA y valor de mercado

In [5]:
# Carga ratings FIFA historicos
fifa_path = os.path.join(DATA_DIR, 'sofifa_history.json')
if os.path.exists(fifa_path):
    with open(fifa_path, encoding='utf-8') as f:
        sofifa = json.load(f)
    print(f'FIFA historico: {len(sofifa)} entradas')
else:
    sofifa = {}
    print('No se encontro sofifa_history.json, usando valores por defecto')

# Carga valores de mercado
tm_path = os.path.join(DATA_DIR, 'transfermarkt_history.json')
if os.path.exists(tm_path):
    with open(tm_path, encoding='utf-8') as f:
        transfermarkt = json.load(f)
    print(f'Transfermarkt: {len(transfermarkt)} entradas')
else:
    transfermarkt = {}
    print('No se encontro transfermarkt_history.json')

# Nota: la ingenieria completa se delega a src/feature_engineering.py
# Este notebook muestra la logica; en produccion usa generate_features()
import sys; sys.path.insert(0, '..')
try:
    from src.feature_engineering import generate_features
    df_final = generate_features(df)
    print(f'Features generadas via generate_features(): {df_final.shape}')
except Exception as e:
    print(f'Usando features calculadas manualmente: {e}')
    df_final = df.copy()


FIFA historico: 19 entradas
Transfermarkt: 20 entradas
Populating Advanced Static Features (FIFA/TM)...
Features generadas via generate_features(): (5700, 268)


## 5. Exportación

In [6]:
# Columnas finales
model_features = [
    'Home_Elo_Calc','Away_Elo_Calc','Elo_Diff',
    'Home_Market_Value','Away_Market_Value','Log_Value_Diff',
    'Diff_FIFA_Ova','Diff_FIFA_Mid','Diff_FIFA_Def','Diff_FIFA_Att',
    'Home_Streak_L5','Away_Streak_L5',
    'Home_H2H_L3','Away_H2H_L3',
]
keep_cols = ['Date','Season','HomeTeam','AwayTeam','FTR','Target',
             'B365H','B365D','B365A'] + model_features

export_cols = [c for c in keep_cols if c in df_final.columns]
df_clean = df_final[export_cols].dropna(subset=['Target','B365H','B365D','B365A'])

df_clean.to_csv(OUT_CSV, index=False)
print(f'Guardado: {OUT_CSV}')
print(f'Shape: {df_clean.shape}')
print(f'NaN por feature:')
print(df_clean[model_features].isna().sum())


Guardado: ../data/df_final_clean.csv
Shape: (5700, 23)
NaN por feature:
Home_Elo_Calc        0
Away_Elo_Calc        0
Elo_Diff             0
Home_Market_Value    0
Away_Market_Value    0
Log_Value_Diff       0
Diff_FIFA_Ova        0
Diff_FIFA_Mid        0
Diff_FIFA_Def        0
Diff_FIFA_Att        0
Home_Streak_L5       0
Away_Streak_L5       0
Home_H2H_L3          0
Away_H2H_L3          0
dtype: int64
